# Sesion 5 - Chatbot RAG Empresarial con PDFs

En este notebook extendemos el asistente RAG de la sesion 4 para trabajar con documentos PDF, ChromaDB y evaluacion simple de grounding.

## Preparacion

Antes de ejecutar la practica, descarga uno o dos PDFs publicos de fuentes como World Bank Open Knowledge Repository, OCDE o Stanford AI Index. Guarda los archivos en una carpeta local y actualiza `document_paths`.

In [ ]:
from pathlib import Path

from agents.rag_course_assistant.agent import RAGCourseAssistant
from agents.rag_course_assistant.evaluation import evaluate_answer
from agents.rag_course_assistant.vector_store import ChromaCourseVectorStore, OllamaEmbeddingFunction

PROJECT_ROOT = Path.cwd()
document_paths = [
    # PROJECT_ROOT / "data" / "sesion5_public_docs" / "reporte.pdf",
]
document_paths

## 1. Crear chunks desde PDFs

El asistente detecta automaticamente documentos `.pdf` y conserva metadatos de pagina para cada chunk.

In [ ]:
store = ChromaCourseVectorStore(
    collection_name="sesion5_rag_pdf_chatbot_notebook",
    embedding_function=OllamaEmbeddingFunction("nomic-embed-text"),
)

assistant = RAGCourseAssistant(
    model_name="qwen2.5:3b",
    embedding_model_name="nomic-embed-text",
    document_path=None,
    document_paths=document_paths,
    vector_store=store,
)

chunks = assistant.build_chunks(max_chars=1200)
len(chunks), chunks[:2]

## 2. Indexar la base vectorial

In [ ]:
indexed = assistant.index_course_content(reset=True)
indexed

## 3. Recuperar evidencia

In [ ]:
question = "Que recomendaciones del documento serian utiles para una oficina de BI?"
retrieved = store.query(question, top_k=5)

for chunk in retrieved:
    print(chunk.chunk_id, Path(chunk.source).name, chunk.page, chunk.distance)
    print(chunk.text[:500])
    print("---")

## 4. Responder con RAG

In [ ]:
response = assistant.answer(question, top_k=5)
print(response.content)

## 5. Evaluar grounding

In [ ]:
evaluation = evaluate_answer(question, response.content, response.retrieved_chunks)
evaluation

## 6. Ejecutar la app

Para usar la interfaz conversacional:

```bash
streamlit run apps/sesion5_rag_chatbot.py
```